## Confidence estimation 方法

In [1]:
from gliner import GLiNER

# Initialize GLiNER with the base model
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

# Sample text for entity prediction
text = """
Cristiano Ronaldo dos Santos Aveiro (Portuguese pronunciation: [kɾiʃˈtjɐnu ʁɔˈnaldu]; born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains both Saudi Pro League club Al Nassr and the Portugal national team. Widely regarded as one of the greatest players of all time, Ronaldo has won five Ballon d'Or awards,[note 3] a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes, the most by a European player. He has won 33 trophies in his career, including seven league titles, five UEFA Champions Leagues, the UEFA European Championship and the UEFA Nations League. Ronaldo holds the records for most appearances (183), goals (140) and assists (42) in the Champions League, goals in the European Championship (14), international goals (128) and international appearances (205). He is one of the few players to have made over 1,200 professional career appearances, the most by an outfield player, and has scored over 850 official senior career goals for club and country, making him the top goalscorer of all time.
"""

# Labels for entity prediction
# Most GLiNER models should work best when entity types are in lower case or title case
labels = ["Person", "Award", "Date", "Competitions", "Teams"]

# Perform entity prediction
entities = model.predict_entities(text, labels, threshold=0.5)

# Display predicted entities and their labels
for entity in entities:
    print(entity["text"], "=>", entity["label"])

/home/tuo96248/anaconda3/envs/scikg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:04<00:00,  1.24it/s]
/home/tuo96248/anaconda3/envs/scikg/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Cristiano Ronaldo dos Santos Aveiro => Person
5 February 1985 => Date
Portugal national team => Teams
Ballon d'Or => Award
UEFA Men's Player of the Year Awards => Award
European Golden Shoes => Award
UEFA Champions Leagues => Competitions
UEFA European Championship => Competitions
UEFA Nations League => Competitions
European Championship => Competitions


In [2]:
entities

[{'start': 1,
  'end': 36,
  'text': 'Cristiano Ronaldo dos Santos Aveiro',
  'label': 'Person',
  'score': 0.8645558953285217},
 {'start': 92,
  'end': 107,
  'text': '5 February 1985',
  'label': 'Date',
  'score': 0.985105037689209},
 {'start': 233,
  'end': 255,
  'text': 'Portugal national team',
  'label': 'Teams',
  'score': 0.5406014919281006},
 {'start': 338,
  'end': 349,
  'text': "Ballon d'Or",
  'label': 'Award',
  'score': 0.6045859456062317},
 {'start': 381,
  'end': 417,
  'text': "UEFA Men's Player of the Year Awards",
  'label': 'Award',
  'score': 0.8173690438270569},
 {'start': 428,
  'end': 449,
  'text': 'European Golden Shoes',
  'label': 'Award',
  'score': 0.8093947768211365},
 {'start': 556,
  'end': 578,
  'text': 'UEFA Champions Leagues',
  'label': 'Competitions',
  'score': 0.8361243605613708},
 {'start': 584,
  'end': 610,
  'text': 'UEFA European Championship',
  'label': 'Competitions',
  'score': 0.8699513077735901},
 {'start': 619,
  'end': 638,
  'te

In [2]:
from openai import OpenAI

def llm_call(prompt: str, system_prompt: str = "", model_name="deepseek-chat") -> str:
    """
    Calls the model with the given prompt and returns the response.

    Args:
        prompt (str): The user prompt to send to the model.
        system_prompt (str, optional): The system prompt to send to the model. Defaults to "".
        model (str, optional): The model to use for the call. Defaults to "claude-3-5-sonnet-20241022".

    Returns:
        str: The response from the language model.
    """
    client = OpenAI(api_key="YOUR_API_KEY", base_url="https://api.deepseek.com") # TODO: Change the API key

    response = client.chat.completions.create(
        model=model_name, # deepseek-chat
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=4096,
        stream=False,
        logprobs=True
    )
    return response.choices[0].message.content, response

In [3]:
prompt = "Please using <entity> to tag entities in the following sentence: Temple University is located in Philadelphia."

# call the model
ans, response = llm_call(prompt, system_prompt="You are a helpful assistant.")
# print(response)

In [5]:
print(ans)

Here is the sentence with entities tagged using `<entity>`:

`<entity>Temple University</entity> is located in <entity>Philadelphia</entity>.`  

- **Temple University** is tagged as an entity (likely an organization/educational institution).  
- **Philadelphia** is tagged as an entity (a location/city).  

Let me know if you'd like additional entity types or adjustments!


In [25]:
response.choices[0].logprobs.content[15]

ChatCompletionTokenLogprob(token='>T', bytes=[62, 84], logprob=-7.15256e-07, top_logprobs=[])

In [26]:
response.choices[0].logprobs.content[0].logprob
logprobs = response.choices[0].logprobs.content[15].logprob
import numpy as np
probs = np.exp(logprobs)
print(probs)

0.9999992847442558


In [ ]:
# convert the logprobs to real probabilities
import numpy as np
import torch
import torch.nn.functional as F

def convert_logprobs_to_probs(logprobs):
    """
    Convert log probabilities to probabilities.

    Args:
        logprobs (list): List of log probabilities.

    Returns:
        list: List of probabilities.
    """
    # Convert log probabilities to probabilities
    probs = np.exp(logprobs)
    return probs


### CrossNER

In [1]:
import json
import os
from collections import Counter
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1

In [12]:
for i in [0, 1, 10, 25, 50, 100]:
    data = load_json(f"./output/politics/deepseek-chat_{i}.json")
    counts = Counter()
    for sent in data:
        if "response" not in sent:
            continue
        pred = extract_entities(sent['response'])
        ref = sent['entities']
        counts = evaluate_sent(ref, pred, counts)
    scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
    print(f"======= {i} =======")
    print(scores_ner)

======= 0 =======
{'precision': 0.6706408345752608, 'recall': 0.6987577639751553, 'f1': 0.6844106463878327}
======= 1 =======
{'precision': 0.6859838274932615, 'recall': 0.7671439336850038, 'f1': 0.7242974030594096}
======= 10 =======
{'precision': 0.7917304747320061, 'recall': 0.7923371647509578, 'f1': 0.7920337035618535}
======= 25 =======
{'precision': 0.8205329153605015, 'recall': 0.8342629482071713, 'f1': 0.8273409719478466}
======= 50 =======
{'precision': 0.8321285140562249, 'recall': 0.8294635708566853, 'f1': 0.830793905372895}
======= 100 =======
{'precision': 0.8414634146341463, 'recall': 0.8374583531651595, 'f1': 0.839456106870229}


{'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [4]:
print(scores_ner)

{'precision': 0.7464935064935065, 'recall': 0.7943615257048093, 'f1': 0.7696839850026781}


In [13]:
data[0]['entities']

[['naive Bayes classifier', 'algorithm'],
 ['Gaussian mixture model', 'algorithm'],
 ['variational autoencoders', 'algorithm']]

### Testing Domain-Specific NER

Datasets includes: 
- climate: ./datasets/climate/

#### Climate

Each json file denotes one labeled document. Each document has several chunks and each chunk has following format:
```json
"text": // the text
"span": // the chunk span in the original doc
"entities": // an entity list
```
each `entity` in ``entities`` has following format:
```json
{
    "begin": 36,
    "end": 46,
    "label": "variable",
    "substring": "deep water",
    "identifier": "https://gcmd.earthdata.nasa.gov/kms/concept/4c4878cb-18f5-464c-b4bb-858273a48d6a"
}
```


In [4]:
import json
import os
from tqdm import tqdm, trange

def read_json_file(json_path):
    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

def get_ent_labels(chunk):
    ent_labels = []
    for ent in chunk["entities"]:
        ent_labels.append(ent["label"])
    return ent_labels
CLIMATE_LABELS = [
        "project",
        "location",
        "model",
        "experiment",
        "platform",
        "instrument",
        "provider",
        "variable",
        "weather event",
        "natural hazard",
        "teleconnection",
        "ocean circulation",
    ]

In [9]:
doc_files = os.listdir('./datasets/climate/')
doc_file = doc_files[0]
data = read_json_file(f'./datasets/climate/{doc_file}')
# datasets/climate_unchunked
old_data = read_json_file(f'./datasets/climate_unchunked/{doc_file}')

In [12]:
data[0]

{'text': "<heading>ABSTRACT</heading>\nFuture energy demand is likely to increase due to climate change, but the magnitude depends on many interacting sources of uncertainty. We combine econometrically estimated responses of energy use to income, hot and cold days with future projections of spatial population and national income under five socioeconomic scenarios and temperature increases around 2050 for two emission scenarios simulated by 21 Earth System Models (ESMs). Here we show that, across 210 realizations of socioeconomic and climate scenarios, vigorous (moderate) warming increases global climate-exposed energy demand before adaptation around 2050 by 25-58% (11-27%), on top of a factor 1.7-2.8 increase above present-day due to socioeconomic developments. We find broad agreement among ESMs that energy demand rises by more than 25% in the tropics and southern regions of the USA, Europe and China. Socioeconomic scenarios vary widely in the number of people in lowincome countries ex

In [6]:
text = data[0]['text']
ent_labels = data[0]['entities']

In [7]:
text

"<heading>ABSTRACT</heading>\nFuture energy demand is likely to increase due to climate change, but the magnitude depends on many interacting sources of uncertainty. We combine econometrically estimated responses of energy use to income, hot and cold days with future projections of spatial population and national income under five socioeconomic scenarios and temperature increases around 2050 for two emission scenarios simulated by 21 Earth System Models (ESMs). Here we show that, across 210 realizations of socioeconomic and climate scenarios, vigorous (moderate) warming increases global climate-exposed energy demand before adaptation around 2050 by 25-58% (11-27%), on top of a factor 1.7-2.8 increase above present-day due to socioeconomic developments. We find broad agreement among ESMs that energy demand rises by more than 25% in the tropics and southern regions of the USA, Europe and China. Socioeconomic scenarios vary widely in the number of people in lowincome countries exposed to 

In [22]:
from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_base")
labels = CLIMATE_LABELS
entities = model.predict_entities(text, labels)
chunk_pred = [(ent['text'], ent['label']) for ent in entities]
chunk_gt = [(ent['substring'], ent['label']) for ent in ent_labels]

# 把chunk_pred和chunk_gt都存起来，然后按照doc存成json文件，最后计算结果。

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 31956.60it/s]
/home/tuo96248/anaconda3/envs/scikg/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [35]:
from gliner import GLiNER

def merge_entities(entities):
    if not entities:
        return []
    merged = []
    current = entities[0]
    for next_entity in entities[1:]:
        if next_entity['label'] == current['label'] and (next_entity['start'] == current['end'] + 1 or next_entity['start'] == current['end']):
            current['text'] = text[current['start']: next_entity['end']].strip()
            current['end'] = next_entity['end']
        else:
            merged.append(current)
            current = next_entity
    # Append the last entity
    merged.append(current)
    return merged


model = GLiNER.from_pretrained("numind/NuNerZero")

# NuZero requires labels to be lower-cased!
labels = CLIMATE_LABELS
# labels = [l.lower() for l in labels]

entities = model.predict_entities(text, labels)

entities = merge_entities(entities)

chunk_pred = [(ent['text'], ent['label']) for ent in entities]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 40986.68it/s]
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [36]:
chunk_pred

[('Earth System Models', 'model'),
 ('tropics', 'location'),
 ('USA', 'location'),
 ('Europe', 'location'),
 ('China', 'location'),
 ('lowincome countries', 'location')]

### process file to jo

In [7]:
# the original datasets
# datasets/climate
files = os.listdir('./datasets/climate/')

In [10]:
new_doc = {}
for file in files:
    output_file = "" + file
    input_data = read_json_file(f'./datasets/climate/{file}')
    output_data = read_json_file(f'./output/baselines/nuNER/{output_file}')
    preds = output_data['pred']
    gts = output_data['gt']
    # iterate over the input data
    for i in range(len(input_data)):
        chunk_labels = input_data[i]['entities']
        chunk_gt = [[ent['substring'], ent['label']] for ent in chunk_labels]
        assert chunk_gt == gts[i]
        span = input_data[i]['span']
        key = (span[0], span[1])
        new_doc[str(key)] = {
            'text': input_data[i]['text'],
            'span': span,
            'entities': chunk_labels,
            'chunk_gt': chunk_gt,
            'chunk_pred': preds[i]
        }
    # save the new doc
    save_json(new_doc, f'./output/baselines/nuNER_s/{file}')

In [51]:
input_data[0].keys()

dict_keys(['text', 'span', 'entities'])

In [44]:
chunk_gt

[('temperature', 'variable'),
 ('Earth System Models', 'model'),
 ('ESMs', 'model'),
 ('ESMs', 'model'),
 ('tropics', 'location'),
 ('USA', 'location'),
 ('temperatures', 'variable'),
 ('irrigation', 'variable'),
 ('cold days', 'natural hazard'),
 ('Europe', 'location'),
 ('China', 'location')]

In [35]:
len(output_data['pred'])

33

In [8]:
# output/baselines/gpt4o_few
gpt_files = os.listdir('output/baselines/ds_few/')

### Performance Computation

In [41]:
import json
import os
from collections import Counter
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1

def read_json_file(json_path):
    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

def safe_div(num, denom):
    if denom > 0:
        return num / denom
    else:
        return 0


def compute_f1(predicted, gold, matched):
    # F1 score.
    precision = safe_div(matched, predicted)
    recall = safe_div(matched, gold)
    f1 = safe_div(2 * precision * recall, precision + recall)
    return dict(precision=precision, recall=recall, f1=f1)


def evaluate_sent(gt_ner, pred_ner,counts):
    # correct_ner = set()
    # Entities.
    counts["ner_gold"] += len(gt_ner)
    counts["ner_predicted"] += len(pred_ner)
    for prediction in pred_ner:
        if any([prediction == actual for actual in gt_ner]):
            counts["ner_matched"] += 1
            # correct_ner.add(prediction[0])
    return counts
def eval_doc(preds, gts, counts):
    for pred, gt in zip(preds, gts):
        counts = evaluate_sent(gt, pred, counts)
    return counts

In [57]:
# NER-DAug/output/baselines/gliner

from collections import Counter
counts = Counter()
# output/baselines/gpt4o_few
setting = "o3-mini_zero_shot"
file_list = os.listdir(f'output/baselines/{setting}/')
# file_list = file_list[:17]
print(len(file_list))
for file in file_list:
    # output/baselines/gpt-4o_zero-shot-CoT
    data = read_json_file(f'output/baselines/{setting}/{file}')
    preds = data['pred']
    gts = data['gt']
    counts = eval_doc(preds, gts, counts)
scores_ner = compute_f1(
            counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
print(scores_ner)

25
{'precision': 0.5852035288655006, 'recall': 0.30936017018491246, 'f1': 0.40475298399614623}


In [93]:
# Function to extract and parse JSON content from Markdown format
def parse_markdown_json(md_string):
    try:
        # Remove the Markdown formatting
        if md_string.startswith("```") and md_string.endswith("```"):
            json_content = md_string.split("\n", 1)[1].rsplit("\n", 1)[0]
        else:
            json_content = md_string  # If no markdown formatting exists

        # Parse the JSON content
        parsed_data = json.loads(json_content)
        return parsed_data
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        return None

def convert_map(parsed_pred):
    pred = {}
    for p in parsed_pred:
        pred[p['entity']] = p['label']
    return pred
def conver_map_label(ents):
    gt = {}
    for e in ents:
        gt[e['substring']] = e['label']
    return gt

def convert_label(ents):
    labels = []
    for ent in ents:
        labels.append([ent['substring'], ent['label']])
    return labels
def convert_label_pred(pred):
    labels = []
    for ent in pred:
        labels.append([ent['entity'], ent['type']])
    return labels

In [40]:
import json
def process_doc(data):
    preds = []
    gts = []
    for i in range(len(data)):
        parsed_pred = parse_markdown_json(data[i]['response'])
        pred = convert_label_pred(parsed_pred)
        gt = convert_label(data[i]['entities'])
        # if len(pred) == len(gt):
        preds.append(pred)
        gts.append(gt)
        # preds.append(pred)
        # gts.append(data[i]['labels'])
    # if len(preds) != len(gts):
    #     print('error')
    #     # make them the same length
    #     preds
    return preds, gts

# entity_typing_results
file_list = os.listdir('entity_typing_results/')
# doc = read_json_file(f'entity_typing_results/{file_list[0]}')
all_preds = []
all_gts = []
error = 0
for file in file_list:
    data = read_json_file(f'entity_typing_results/{file}')
    preds, gts = process_doc(data)
    all_preds.extend(preds)
    all_gts.extend(gts)

NameError: name 'parse_markdown_json' is not defined

In [103]:
def eval_doc(preds, gts, counts):
    for pred, gt in zip(preds, gts):
        counts = evaluate_sent(gt, pred, counts)
    return counts
count = Counter()
for doc_preds, doc_gts in zip(all_preds, all_gts):
    for chunk_pred, chunk_gt in zip(doc_preds, doc_gts):
        count = evaluate_sent(chunk_gt, chunk_pred, count)
scores_ner = compute_f1(
            counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
print(scores_ner)

{'precision': 0.3620636707953731, 'recall': 0.3957703927492447, 'f1': 0.37816743290922694}


In [74]:
import json
def process_doc(data):
    preds = []
    gts = []
    for i in range(len(data)):
        parsed_pred = parse_markdown_json(data[i]['response'])
        pred = [p['type'] for p in parsed_pred]
        gt = [e['label'] for e in data[i]['entities']]
        if len(pred) == len(gt):
            preds.append(pred)
            gts.append(gt)
        # preds.append(pred)
        # gts.append(data[i]['labels'])
    # if len(preds) != len(gts):
    #     print('error')
    #     # make them the same length
    #     preds
    return preds, gts

# entity_typing_results
file_list = os.listdir('entity_typing_results/')
# doc = read_json_file(f'entity_typing_results/{file_list[0]}')
all_preds = []
all_gts = []
error = 0
for file in file_list:
    data = read_json_file(f'entity_typing_results/{file}')
    preds, gts = process_doc(data)
    all_preds.extend(preds)
    all_gts.extend(gts)

In [84]:
from sklearn.metrics import precision_score, recall_score, f1_score

def calculate_metrics(pred, label):
    """
    计算 Precision, Recall 和 F1 分数
    
    参数:
    - pred: 预测的标签列表
    - label: 实际的标签列表
    
    返回:
    - 一个字典，包含 Precision, Recall 和 F1
    """
    if len(pred) != len(label):
        raise ValueError("预测和标签列表长度不一致")
    
    # 计算 Precision, Recall, 和 F1
    precision = precision_score(label, pred, average='macro')
    recall = recall_score(label, pred, average='macro')
    f1 = f1_score(label, pred, average='macro')
    # 计算accuracy
    accuracy = sum([1 for i in range(len(pred)) if pred[i] == label[i]]) / len(pred)
    
    return {
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Accuracy': accuracy
    }

In [75]:
# flatten all_preds and all_gts
all_preds = [item for sublist in all_preds for item in sublist]
all_gts = [item for sublist in all_gts for item in sublist]

In [81]:
# replace None to string 'None' in all_preds and all_gts
all_preds = ['None' if x is None else x for x in all_preds]
all_gts = ['None' if x is None else x for x in all_gts]

In [82]:
len(all_preds), len(all_gts)

(11312, 11312)

In [85]:
metrics = calculate_metrics(all_preds, all_gts)
print(metrics)

{'Precision': 0.4071561216554777, 'Recall': 0.3641979642919279, 'F1': 0.3557195746003971, 'Accuracy': 0.6238507779349364}


/home/tuo96248/anaconda3/envs/NER/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/tuo96248/anaconda3/envs/NER/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [70]:
print(all_preds[:8])
print(all_gts[:8])

['location', 'model', 'model', 'location', 'variable', 'location', 'variable', 'model']
['location', 'model', 'model', 'location', 'variable', 'location', 'variable', 'model']


In [51]:
ents = data[0]['entities']
labels = [ent['label'] for ent in ents]

In [25]:
import json

# Example JSON strings
json_string_with_data = '''
[
    {"entity": "temperature", "type": "variable"},
    {"entity": "radiative forcing", "type": "variable"},
    {"entity": "Intergovernmental Panel on Climate Change", "type": "provider"},
    {"entity": "IPCC", "type": "provider"},
    {"entity": "Representative Concentration Pathways", "type": "experiment"},
    {"entity": "RCPs", "type": "experiment"},
    {"entity": "Shared Socioeconomic Pathways", "type": "experiment"},
    {"entity": "Shared Socioeconomic Pathway", "type": "experiment"},
    {"entity": "SSPs", "type": "experiment"}
]
'''

json_string_empty = '[]'

# Function to parse and handle JSON
def parse_json(json_string):
    try:
        parsed_data = json.loads(json_string)
        if not parsed_data:  # Check if the list is empty
            print("The JSON is empty.")
        else:
            for item in parsed_data:
                print(f"Entity: {item['entity']}, Type: {item['type']}")
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")

# Test with both JSON strings
print("Parsing JSON with data:")
parse_json(json_string_with_data)

print("\nParsing empty JSON:")
parse_json(json_string_empty)

Parsing JSON with data:
Entity: temperature, Type: variable
Entity: radiative forcing, Type: variable
Entity: Intergovernmental Panel on Climate Change, Type: provider
Entity: IPCC, Type: provider
Entity: Representative Concentration Pathways, Type: experiment
Entity: RCPs, Type: experiment
Entity: Shared Socioeconomic Pathways, Type: experiment
Entity: Shared Socioeconomic Pathway, Type: experiment
Entity: SSPs, Type: experiment

Parsing empty JSON:
The JSON is empty.


In [34]:
import json
import os
from tqdm import tqdm, trange
import re


def parse_llm_response_with_codeblock(json_str):
    """
    Parses a string containing a code block fenced by triple backticks (```),
    which in turn contains valid JSON. It then extracts and converts the JSON
    to a list of [text, type] entries.
    
    :param json_str: The multi-line string that includes ```json ... ```,
                     e.g.:
                     \"\"\"```json
                     [
                         {"text": "flow duration curve", "type": "variable"},
                         ...
                     ]
                     ```\"\"\"
    :return: A list of lists, where each inner list is [text, type].
    """
    # 1) Remove the triple backtick fences (and "json" marker if present).
    #    This regular expression will remove lines like:
    #       ```json
    #       ```
    #    from the string.
    cleaned_str = re.sub(r'```(?:json)?', '', json_str).strip('`\n ')
    
    # 2) Now 'cleaned_str' should contain just the raw JSON array. Let's parse it.
    try:
        data = json.loads(cleaned_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON data after removing code fences: {e}")
    
    # 3) Convert each object in the JSON array to [text, type] format.
    if not isinstance(data, list):
        raise ValueError("The extracted JSON is not a list at the top level.")
    
    result = []
    for item in data:
        text = item.get("text")
        type_ = item.get("type")
        # Optionally, add checks here if text or type might be missing.
        result.append([text, type_])
    
    return result

def get_parsed_preds(preds):
    parsed_preds = []
    for pred in preds:
        try:
            res = parse_llm_response_with_codeblock(pred)
        except:
            res = []
            print("parsed error + 1")
        parsed_preds.append(res)
    return parsed_preds

In [25]:
parsed = parse_llm_response_with_codeblock(preds[0])

In [ ]:
# NER-DAug/output/baselines/gliner

from collections import Counter
counts = Counter()
file_list = os.listdir('./output/baselines/nuNER/')
for file in file_list:
    data = read_json_file(f'./output/baselines/nuNER/{file}')
    preds = data['pred']
    gts = data['gt']
    counts = eval_doc(preds, gts, counts)
scores_ner = compute_f1(
            counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
print(scores_ner)

In [15]:
# NER-DAug/output/baselines/gliner

from collections import Counter
counts = Counter()
file_list = os.listdir('./output/baselines/nuNER/')
for file in file_list:
    data = read_json_file(f'./output/baselines/nuNER/{file}')
    preds = data['pred']
    gts = data['gt']
    counts = eval_doc(preds, gts, counts)
scores_ner = compute_f1(
            counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
print(scores_ner)

{'precision': 0.502455690796498, 'recall': 0.19252168221240387, 'f1': 0.27837917775805976}


In [5]:
data = read_json_file(f'./output/baselines/gliner/{file_list[0]}')

In [7]:
preds = data['pred']
gts = data['gt']

{'precision': 0.389937106918239, 'recall': 0.09337349397590361, 'f1': 0.15066828675577154}


In [ ]:
text

'<heading>ABSTRACT</heading>\nFuture energy demand is likely to increase due to climate change, but the magnitude depends on many interacting sources of uncertainty. We combine econometrically estimated responses of energy use to income, hot and cold days with future projections of spatial population and national income under five socioeconomic scenarios and temperature increases around 2050 for two emission scenarios simulated by 21 Earth System Models (ESMs). Here we show that, across 210 realizations of socioeconomic and climate scenarios, vigorous (moderate) warming increases global climate-exposed energy demand before adaptation around 2050 by 25-58% (11-27%), on top of a factor 1.7-2.8 increase above present-day due to socioeconomic developments. We find broad agreement among ESMs that energy demand rises by more than 25% in the tropics and southern regions of the USA, Europe and China. Socioeconomic scenarios vary widely in the number of people in lowincome countries exposed to 

In [ ]:
from nltk.tokenize import sent_tokenize

def split_into_sentences(text):
    sentences = sent_tokenize(text)  # 使用nltk进行句子切分
    return sentences
def get_sentence_positions(text):
    sentences = split_into_sentences(text)
    positions = []
    start = 0
    for sentence in sentences:
        start_idx = text.find(sentence, start)  # 找到句子的起始位置
        end_idx = start_idx + len(sentence)
        positions.append((start_idx, end_idx))
        start = end_idx
    return sentences, positions
# 为了适应RoBERTa的输入最大长度，我们需要对文本进行切分，然后每个chunk包含多个句子，总的长度不超过max_length。即总的长度不能超过1500
# 包含至少3个句子。然后我们要把原始的实体位置映射到新的chunk上。
def get_chunks(sentences, positions, max_length=1536):
    chunks = []
    chunk = []
    chunk_positions = []
    chunk_length = 0
    for sentence, position in zip(sentences, positions):
        sentence_length = len(sentence)
        if chunk_length + sentence_length < max_length:
            chunk.append(sentence)
            chunk_positions.append(position)
            chunk_length += sentence_length
        else:
            chunks.append((chunk, chunk_positions))
            chunk = [sentence]
            chunk_positions = [position]
            chunk_length = sentence_length
    if chunk:
        chunks.append((chunk, chunk_positions))
    return chunks
def find_entities(chunk_start, chunk_end, entities):
    selected_entities = []
    for entity in entities:
        entity_start = entity["begin"]
        entity_end = entity["end"]
        if entity_start >= chunk_start and entity_end <= chunk_end:
            selected_entities.append(entity)
    return selected_entities

def update_chunk_entity_spans(start_position, entities):
    updated_entities = []
    for entity in entities:
        entity["begin"] = entity["begin"] - start_position
        entity["end"] = entity["end"] - start_position
        updated_entities.append(entity)
    return updated_entities

In [ ]:
chunked_files = os.listdir("./datasets/climate_chunk/")
for file in chunked_files:
    data = read_json_file(f"./datasets/climate_chunk/{file}")
    for chunk in data:
        text = chunk["text"]
        entities = chunk["entities"]
        for entity in entities:
            entity_text = text[entity["begin"]:entity["end"]]
            assert entity_text == entity["substring"], f"{entity_text} != {entity['substring']}"

In [ ]:
# 遍历每个chunk，找到实体在chunk中的位置，然后映射到原始文本中的位置。更新实体的位置。
# 1. 先找到一个chunk中所有的实体
doc = []
for chunk in chunks:
    end_position = chunk[1][-1][1]
    start_position = chunk[1][0][0]
    entities_in_chunk = find_entities(start_position, end_position, entities)
    chunk_entities = update_chunk_entity_spans(start_position, entities_in_chunk)
    chunk_span = (start_position, end_position)
    chunk_text = text[start_position:end_position]
    doc.append({
        "text": chunk_text,
        "span": chunk_span,
        "entities": chunk_entities,
    })

In [ ]:
print(sentences[1])
print(positions[1])
print(text[positions[1][0]:positions[1][1]])

We combine econometrically estimated responses of energy use to income, hot and cold days with future projections of spatial population and national income under five socioeconomic scenarios and temperature increases around 2050 for two emission scenarios simulated by 21 Earth System Models (ESMs).
(164, 463)
We combine econometrically estimated responses of energy use to income, hot and cold days with future projections of spatial population and national income under five socioeconomic scenarios and temperature increases around 2050 for two emission scenarios simulated by 21 Earth System Models (ESMs).


In [ ]:
print(sentences[-1])
print(positions[-1])

Publisher's note: Springer Nature remains neutral with regard to jurisdictional claims in published maps and institutional affiliations.
(46408, 46544)


In [ ]:
len(text)

46545

In [ ]:
text[-100:-1]

'ains neutral with regard to jurisdictional claims in published maps and institutional affiliations.'

In [ ]:
text[-1]

'\n'